# Train Basic Hairstyle Attribute Predictor

Lightweight multi-head classifier for the reviewed kept-asset basic fields: `length` and `curl`.

In [16]:
import json
from pathlib import Path
import sys
import time

import pandas as pd

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'backend').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not locate project root.')

def format_seconds(seconds: float) -> str:
    total_seconds = max(0, int(seconds))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    if hours:
        return f'{hours:d}:{minutes:02d}:{secs:02d}'
    return f'{minutes:02d}:{secs:02d}'

def resolve_training_device(torch_module, require_gpu: bool = True) -> str:
    if torch_module.cuda.is_available():
        return 'cuda'
    if require_gpu:
        raise RuntimeError(
            'CUDA GPU is required for this training run, but the current PyTorch build does not have CUDA available. '
            'Install a CUDA-enabled PyTorch build into .venv and restart the notebook kernel.'
        )
    return 'cpu'

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
BACKEND_ROOT = PROJECT_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.append(str(BACKEND_ROOT))

import torch
from torch.utils.data import DataLoader

from app.ml.datasets import MultiAttributeDataset, read_jsonl_manifest
from app.ml.hairstyle_attribute_model import build_attribute_model, multitask_cross_entropy
from app.ml.metrics import attribute_accuracy, exact_match_accuracy
from app.ml.transforms import ResizeImage

PROJECT_ROOT

WindowsPath('D:/Projects/Personal Projects/Hairstyle Recommender Live Tryon')

In [17]:
DATASET_ROOT = PROJECT_ROOT / 'backend' / 'data' / 'datasets' / 'reviewed_hairstyle_basic'
TRAIN_MANIFEST = DATASET_ROOT / 'train.jsonl'
VAL_MANIFEST = DATASET_ROOT / 'val.jsonl'
VOCAB_PATH = DATASET_ROOT / 'label_vocab.json'
SUMMARY_PATH = DATASET_ROOT / 'summary.json'
CHECKPOINT_PATH = PROJECT_ROOT / 'backend' / 'checkpoints' / 'reviewed_hairstyle_basic' / 'basic_attribute_model.pt'

REQUIRE_GPU = True
IMAGE_SIZE = 224
BATCH_SIZE = 8
EPOCHS = 5
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
DEVICE = resolve_training_device(torch, require_gpu=REQUIRE_GPU)

CORE_FIELDS = ('length', 'curl')


In [18]:
train_records = read_jsonl_manifest(TRAIN_MANIFEST)
val_records = read_jsonl_manifest(VAL_MANIFEST)
label_vocab = json.loads(VOCAB_PATH.read_text(encoding='utf-8'))
summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))

attribute_vocab_sizes = {field: len(label_vocab[field]) for field in CORE_FIELDS}

print('Device:', DEVICE)
print('Torch version:', torch.__version__)
print('CUDA version:', torch.version.cuda)
print('Core fields:', CORE_FIELDS)
print('Train records:', len(train_records))
print('Val records:', len(val_records))
summary

Device: cpu
Core fields: ('length', 'curl')
Train records: 67
Val records: 17


{'core_fields': ['length', 'curl'],
 'total_records': 84,
 'train_records': 67,
 'val_records': 17,
 'train_ratio': 0.8,
 'field_value_counts': {'length': {'long': 44, 'short': 35, 'medium': 5},
  'curl': {'straight': 45, 'wavy': 37, 'curly': 2}}}

In [19]:
transform = ResizeImage((IMAGE_SIZE, IMAGE_SIZE))

train_dataset = MultiAttributeDataset(
    train_records,
    label_vocab=label_vocab,
    transform=transform,
    fields=CORE_FIELDS,
)
val_dataset = MultiAttributeDataset(
    val_records,
    label_vocab=label_vocab,
    transform=transform,
    fields=CORE_FIELDS,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

batch = next(iter(train_loader))
batch['image'].shape, {field: tensor.shape for field, tensor in batch['labels'].items()}

(torch.Size([8, 3, 224, 224]),
 {'length': torch.Size([8]), 'curl': torch.Size([8])})

In [20]:
model = build_attribute_model(attribute_vocab_sizes=attribute_vocab_sizes, base_channels=32, dropout=0.2).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

sum(parameter.numel() for parameter in model.parameters())

1174758

In [21]:
def move_targets_to_device(targets, device):
    return {field: tensor.to(device) for field, tensor in targets.items()}

def filter_outputs(outputs, fields):
    return {field: outputs[field] for field in fields}

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    running_loss = 0.0
    total_batches = 0
    for batch in loader:
        images = batch['image'].to(device)
        targets = move_targets_to_device(batch['labels'], device)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        outputs = filter_outputs(outputs, CORE_FIELDS)
        loss = multitask_cross_entropy(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += float(loss.item())
        total_batches += 1
    return running_loss / max(total_batches, 1)

def evaluate(model, loader, device):
    model.eval()
    running_loss = 0.0
    total_batches = 0
    output_batches = []
    target_batches = []

    with torch.no_grad():
        for batch in loader:
            images = batch['image'].to(device)
            targets = move_targets_to_device(batch['labels'], device)
            outputs = model(images)
            outputs = filter_outputs(outputs, CORE_FIELDS)
            loss = multitask_cross_entropy(outputs, targets)
            running_loss += float(loss.item())
            total_batches += 1
            output_batches.append({field: tensor.detach().cpu() for field, tensor in outputs.items()})
            target_batches.append({field: tensor.detach().cpu() for field, tensor in targets.items()})

    merged_outputs = {
        field: torch.cat([batch[field] for batch in output_batches], dim=0)
        for field in CORE_FIELDS
    }
    merged_targets = {
        field: torch.cat([batch[field] for batch in target_batches], dim=0)
        for field in CORE_FIELDS
    }
    attr_scores = attribute_accuracy(merged_outputs, merged_targets)
    return {
        'val_loss': running_loss / max(total_batches, 1),
        'exact_match_accuracy': exact_match_accuracy(merged_outputs, merged_targets),
        **attr_scores,
    }


In [22]:
history = []
best_metric = float('-inf')
best_epoch = None
epoch_durations = []

for epoch in range(1, EPOCHS + 1):
    print(f'Running epoch {epoch}/{EPOCHS}...')
    epoch_start = time.time()
    train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
    metrics = evaluate(model, val_loader, DEVICE)
    epoch_duration = time.time() - epoch_start
    epoch_durations.append(epoch_duration)

    row = {
        'epoch': epoch,
        'train_loss': train_loss,
        'epoch_seconds': round(epoch_duration, 2),
        **metrics,
    }
    history.append(row)

    current_metric = metrics['exact_match_accuracy']
    if current_metric > best_metric:
        best_metric = current_metric
        best_epoch = epoch

    average_epoch_seconds = sum(epoch_durations) / len(epoch_durations)
    remaining_seconds = average_epoch_seconds * (EPOCHS - epoch)

    CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            'model_state_dict': model.state_dict(),
            'history': history,
            'last_epoch': epoch,
            'core_fields': CORE_FIELDS,
            'label_vocab': label_vocab,
        },
        CHECKPOINT_PATH,
    )

    print(f"Completed epoch {epoch}/{EPOCHS}")
    print(f"Epoch time: {format_seconds(epoch_duration)}")
    print(f"Estimated remaining: {format_seconds(remaining_seconds)}")
    print(f"Best epoch so far: {best_epoch} (exact_match_accuracy={best_metric:.4f})")
    print(row)
    display(pd.DataFrame(history))

print(f'Saved checkpoint to {CHECKPOINT_PATH}')
pd.DataFrame(history)

Running epoch 1/5...
Completed epoch 1/5
Epoch time: 00:01
Estimated remaining: 00:05
Best epoch so far: 1 (exact_match_accuracy=0.2941)
{'epoch': 1, 'train_loss': 2.248099856906467, 'epoch_seconds': 1.36, 'val_loss': 2.144345283508301, 'exact_match_accuracy': 0.29411765933036804, 'length': 0.47058823704719543, 'curl': 0.4117647111415863}


,epoch,train_loss,epoch_seconds,val_loss,exact_match_accuracy,length,curl
0,1,2.2481,1.36,2.144345,0.294118,0.470588,0.411765


Running epoch 2/5...
Completed epoch 2/5
Epoch time: 00:01
Estimated remaining: 00:04
Best epoch so far: 1 (exact_match_accuracy=0.2941)
{'epoch': 2, 'train_loss': 1.806241101688809, 'epoch_seconds': 1.31, 'val_loss': 1.56812584400177, 'exact_match_accuracy': 0.1764705926179886, 'length': 0.4117647111415863, 'curl': 0.529411792755127}


,epoch,train_loss,epoch_seconds,val_loss,exact_match_accuracy,length,curl
0,1,2.248100,1.36,2.144345,0.294118,0.470588,0.411765
1,2,1.806241,1.31,1.568126,0.176471,0.411765,0.529412


Running epoch 3/5...
Completed epoch 3/5
Epoch time: 00:01
Estimated remaining: 00:02
Best epoch so far: 3 (exact_match_accuracy=0.4118)
{'epoch': 3, 'train_loss': 1.6412949164708455, 'epoch_seconds': 1.52, 'val_loss': 1.733686884244283, 'exact_match_accuracy': 0.4117647111415863, 'length': 0.6470588445663452, 'curl': 0.529411792755127}


...

,epoch,train_loss,epoch_seconds,val_loss,exact_match_accuracy,length,curl
0,1,2.248100,1.36,2.144345,0.294118,0.470588,0.411765
1,2,1.806241,1.31,1.568126,0.176471,0.411765,0.529412
2,3,1.641295,1.52,1.733687,0.411765,0.647059,0.529412
3,4,1.523528,1.50,1.687888,0.352941,0.647059,0.470588


Running epoch 5/5...
Completed epoch 5/5
Epoch time: 00:01
Estimated remaining: 00:00
Best epoch so far: 3 (exact_match_accuracy=0.4118)
{'epoch': 5, 'train_loss': 1.3471842341952853, 'epoch_seconds': 1.51, 'val_loss': 1.7524444262186687, 'exact_match_accuracy': 0.4117647111415863, 'length': 0.529411792755127, 'curl': 0.7058823704719543}


,epoch,train_loss,epoch_seconds,val_loss,exact_match_accuracy,length,curl
0,1,2.248100,1.36,2.144345,0.294118,0.470588,0.411765
1,2,1.806241,1.31,1.568126,0.176471,0.411765,0.529412
2,3,1.641295,1.52,1.733687,0.411765,0.647059,0.529412
3,4,1.523528,1.50,1.687888,0.352941,0.647059,0.470588
4,5,1.347184,1.51,1.752444,0.411765,0.529412,0.705882


Saved checkpoint to D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon\backend\checkpoints\reviewed_hairstyle_basic\basic_attribute_model.pt


,epoch,train_loss,epoch_seconds,val_loss,exact_match_accuracy,length,curl
0,1,2.248100,1.36,2.144345,0.294118,0.470588,0.411765
1,2,1.806241,1.31,1.568126,0.176471,0.411765,0.529412
2,3,1.641295,1.52,1.733687,0.411765,0.647059,0.529412
3,4,1.523528,1.50,1.687888,0.352941,0.647059,0.470588
4,5,1.347184,1.51,1.752444,0.411765,0.529412,0.705882
